In [ ]:
# Cell 1: 安装依赖（第一次运行时执行）
# !pip install transformers torch pandas tqdm

In [2]:
# Cell 2: 导入
import pandas as pd
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm

In [3]:
# Cell 3: 从 kg.csv 收集所有唯一实体
df = pd.read_csv('../data/kg.csv')

# x端和y端分别处理，再合并
x = df[['x_type', 'x_index', 'x_name']].copy()
x.columns = ['type', 'index', 'name']

y = df[['y_type', 'y_index', 'y_name']].copy()
y.columns = ['type', 'index', 'name']

combined = pd.concat([x, y]).drop_duplicates(subset=['type', 'index'])
combined['id'] = combined['type'] + '::' + combined['index'].astype(str)
combined['name'] = combined['name'].where(combined['name'].notna(), combined['index'].astype(str))

entities = dict(zip(combined['id'], combined['name'].astype(str)))
print(f'共 {len(entities)} 个实体')

/tmp/ipykernel_90849/1247776716.py:2: DtypeWarning: Columns (3,8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/kg.csv')


共 129375 个实体


In [4]:

# Cell 4: 加载 PubMedBERT（从本地路径加载，不联网）
LOCAL_MODEL_PATH = '../models/pubmedbert'  # 改成你实际上传的路径

tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
model = AutoModel.from_pretrained(LOCAL_MODEL_PATH)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.eval().to(device)
print(f'使用设备: {device}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

使用设备: cuda


In [5]:


# Cell 5: 批量编码并保存
BATCH_SIZE = 128  # 显存足够时适当调大，吞吐量更高

ids   = list(entities.keys())
names = list(entities.values())

# 用列表直接存 id 和 embedding，最后一次性建 DataFrame
# 避免每条记录创建一个字典，减少内存分配开销
all_ids  = []
all_embs = []

with torch.no_grad():
    for i in tqdm(range(0, len(ids), BATCH_SIZE)):
        batch_ids   = ids[i : i + BATCH_SIZE]
        batch_names = names[i : i + BATCH_SIZE]

        inputs = tokenizer(
            batch_names,
            padding=True,
            truncation=True,
            max_length=64,
            return_tensors='pt'
        ).to(device)  # 直接链式 .to(device)，省去字典推导

        # 只取最后一层 [CLS]，不保留完整 output 节省显存
        embs = model(**inputs, output_hidden_states=False) \
                   .last_hidden_state[:, 0, :].cpu().numpy()

        all_ids.extend(batch_ids)
        all_embs.append(embs)  # 先攒起来，最后一次 vstack

# 一次性拼接所有 embedding，比逐条 append 快很多
all_embs = np.vstack(all_embs)

pd.DataFrame({
    'id': all_ids,
    'embedding': [np.array(e, dtype=np.float32).copy() for e in all_embs]
}).to_pickle('../data/pubmedbert_embeddings.pkl')
print(f'完成！共 {len(all_ids)} 条')

100%|██████████| 1011/1011 [00:49<00:00, 20.32it/s]


完成！共 129375 条
